In [2]:
from langgraph.graph import StateGraph, START, END
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from typing import TypedDict, Literal, Annotated
from langchain_core.messages import SystemMessage, HumanMessage
import os

In [3]:
load_dotenv()
api_key = os.getenv("GOOGLE_API_KEY")

In [4]:
llm = ChatGoogleGenerativeAI(
    model = 'gemini-3.8-flash',
    api_key = api_key
)

In [5]:
class TweetState(TypedDict):

    topic: str
    tweet: str
    evaluation: Literal['approved',"needs_improvement"]
    feedback:str
    iteration:int
    max_iteration = int

In [ ]:
from pydantic import BaseModel, Field

class TweetsEValuation(BaseModel):
    evaluation: Literal['approved', 'needs_improvement'] = Field(..., description='FInal evaluation of the ttweet')
    feedback =  str = Field(..., description='Contrusctive feedback for the tweet')

In [ ]:
def generate_tweet(state: TweetState):

    messages = [
        SystemMessage(content='You are a funny and clever TWitter/X influencer'),
        HumanMessage(content=f""" Wirte a short, original, and hilarious tweet on the topic:"{state['topic']}. 
        RUles: 
        -Do NOt use question-answer format, max 280 characters, use observational humor, irony, sarcasm, or cultural refrences""")
    ]
    response = llm.invoke(messages).content
    return {'tweet': response}
    

In [ ]:
def evaluation(state:TweetState):
    messages = [
        SystemMessage(content='You are a tweet evalluator'),
        HumanMessage(content=f'''evalaute the tweet: {state['tweet']}, reepsonds only  in approved or needs_improvement, nothhing else just this''')
    ]

    evaluated = llm.invoke(messages).content

    return {
        'evaluation'
    }


In [ ]:
graph = StateGraph(TweetState)

graph.add_node('generate', generate_tweet)
graph.add_node('evaluation', evaluation)
graph.add_node('opt', opt)
